# QQ 图与正态性

> 对应代码：`MathStatsCode/code_in_notes/Chap.14/qqplot_hs300.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebook/PyStata.md`。

**实验目的**：手工计算理论分位数绘制 QQ 图，再与官方命令 qnorm 以及带正态密度曲线的直方图对比。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.14`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。


In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.14")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

`clear` 清空数据，载入沪深300指数数据（含日收益率 retindex）。本 notebook 用 **QQ 图**考察收益率是否服从正态分布。


In [ ]:
%%stata
clear
use ../datasets/hs300index.dta


**计算quantile，为了避免出现0和1，减去了0.5**

`sort retindex` 排序后，`gen q=(_n-0.5)/_N` 计算每个观测的**样本分位点**：排序后的行号 _n 对应经验分位数，减去0.5（或其它修正）是为了避免取到0和1这两个端点值（正态分位函数在端点处发散）。


In [ ]:
%%stata
sort retindex
gen q=(_n-0.5)/_N


**计算均值、标准差**

`sum retindex` 取得样本均值与标准差（存入宏 mu、sd），它们是理论正态分位数的两个参数。


In [ ]:
%%stata
sum retindex
local mu=r(mean)
local sd=r(sd)


**计算正态分布的quantile**

`gen normal_q = sd×invnormal(q)+mu`：`invnormal(q)` 是标准正态分布的**分位函数**（即 Φ⁻¹(q)），乘以 sd 加 mu 后得到与样本分位点对应的**理论正态分位数**。


In [ ]:
%%stata
gen normal_q = `sd'*invnormal(q)+`mu'


**画图**

画 **QQ 图**：以理论正态分位数 `normal_q` 为横轴、样本分位数 `retindex` 为纵轴画散点，并叠加45度参考线 `(line normal_q normal_q)`。若数据服从正态分布，散点应紧贴这条直线；这里可以看到**尾部偏离直线**（厚尾），说明收益率不服从正态分布。`legend(off)` 隐藏图例。


In [ ]:
%%stata
twoway (scatter retindex normal_q) /*
    */(line normal_q normal_q), legend(off)


**官方命令**

用官方命令 `qnorm retindex` 画标准 QQ 图（Stata 自带的正态分位图），与上一步手工实现的结果相互印证；`graph export` 导出。


In [ ]:
%%stata
qnorm retindex


**直方图**

最后 `hist retindex, norm` 画直方图并叠加正态密度曲线（`norm` 选项），从第三个角度观察分布形状；`graph export` 导出——直方图与正态曲线在尾部明显不吻合，同样说明收益率具有厚尾特征。


In [ ]:
%%stata
hist retindex, norm
